### DeepLC Walkthrough with Prospect PTM Data
This example builds DeepLC features from the dataset and trains a DeepLC retention-time prediction model.

In [ ]:
!pip install dlomix

In [ ]:
# install pyteomics to fetch feature values for modifications given their unimod IDs
!pip install pyteomics

DeepLC uses four inputs: sequence ids, atoms per position, di-amino-acid atoms, and optional global features.
The feature functions below generate the tensors expected by the model using the feature extraction flow from DLOmix.

### 0 Constants

In [ ]:
_MAX_LEN  = 60                    # maximum padded sequence length
_N_ATOMS  = 6                     # number of tracked elements: C H N O S P
_PAD_ATOM = [0, 0, 0, 0, 0, 0]   # atom vector used for padding positions
_PAD_TOKEN = '-'   # key in ALPHABET used for padding positions
_PAD_TOKEN_ID = 0  # index in ALPHABET used for padding positions
ATOM_ORDER = ["C", "H", "N", "O", "S", "P"]

# Positive indices: first 4 residues of the peptide
_POS_INDICES = [0, 1, 2, 3]
# Negative indices: last 4 residues (-4 = 4th from end, -1 = last)
_NEG_INDICES = [-4, -3, -2, -1]


# ── Per-residue atomic composition  [C, H, N, O, S, P] ───────────────────────
# RESIDUE composition = amino acid minus H₂O (as it exists in a peptide chain).
_AA_ATOMS = {
    'A': [ 3,  5,  1,  1,  0,  0],  # Alanine
    'R': [ 6, 12,  4,  1,  0,  0],  # Arginine
    'N': [ 4,  6,  2,  2,  0,  0],  # Asparagine
    'D': [ 4,  5,  1,  3,  0,  0],  # Aspartate
    'C': [ 3,  5,  1,  1,  1,  0],  # Cysteine
    'E': [ 5,  7,  1,  3,  0,  0],  # Glutamate
    'Q': [ 5,  8,  2,  2,  0,  0],  # Glutamine
    'G': [ 2,  3,  1,  1,  0,  0],  # Glycine
    'H': [ 6,  7,  3,  1,  0,  0],  # Histidine
    'I': [ 6, 11,  1,  1,  0,  0],  # Isoleucine
    'L': [ 6, 11,  1,  1,  0,  0],  # Leucine
    'K': [ 6, 12,  2,  1,  0,  0],  # Lysine
    'M': [ 5,  9,  1,  1,  1,  0],  # Methionine
    'F': [ 9,  9,  1,  1,  0,  0],  # Phenylalanine
    'P': [ 5,  7,  1,  1,  0,  0],  # Proline
    'S': [ 3,  5,  1,  2,  0,  0],  # Serine
    'T': [ 4,  7,  1,  2,  0,  0],  # Threonine
    'W': [11, 10,  2,  1,  0,  0],  # Tryptophan
    'Y': [ 9,  9,  1,  2,  0,  0],  # Tyrosine
    'V': [ 5,  9,  1,  1,  0,  0],  # Valine
    'U': [ 3,  5,  1,  1,  0,  0],  # Selenocysteine (treated as Cys)
}

_UNIMOD_DELTAS = {
    #  id    C    H    N    O    S    P
       1:  [ 2,  2,  0,  1,  0,  0],  # Acetyl            +C2H2O
       2:  [ 0,  1,  1, -1,  0,  0],  # Amidated          +NH−O  (C-term)
       3:  [10, 15,  2,  2,  1,  0],  # Biotinyl          +C10H15N2O2S
       4:  [ 2,  3,  1,  1,  0,  0],  # Carbamidomethyl   +C2H3NO
       5:  [ 1,  1,  1,  1,  0,  0],  # Carbamyl          +CHNO
       6:  [ 2,  3,  0,  2,  0,  0],  # Carboxymethyl     +C2H3O2
       7:  [ 0, -1, -1,  1,  0,  0],  # Deamidated        −NH+OH
      21:  [ 0,  1,  0,  3,  0,  1],  # Phospho           +H1O3P1
      24:  [ 3,  5,  1,  1,  0,  0],  # Propionamide      +C3H5NO
      26:  [ 2,  1,  1,  0,  0,  0],  # Pyro-carbamidomethyl
      27:  [ 0, -2,  0, -1,  0,  0],  # Pyro-glu (from E) −H2O
      28:  [ 0, -2,  0, -1,  0,  0],  # Pyro-glu (from Q) −H2O (same formula)
      34:  [ 1,  2,  0,  0,  0,  0],  # Methyl            +CH2
      35:  [ 0,  0,  0,  1,  0,  0],  # Oxidation         +O
      36:  [ 2,  4,  0,  0,  0,  0],  # Dimethyl          +C2H4
      37:  [ 3,  6,  0,  0,  0,  0],  # Trimethyl         +C3H6
      40:  [ 0,  0,  0,  3,  1,  0],  # Sulfo             +O3S
      43:  [ 2,  2,  0,  1,  0,  0],  # Acetyl (alt acc.) same as UNIMOD:1
      47:  [16, 30,  0,  1,  0,  0],  # Palmitoyl         +C16H30O
      58:  [ 3,  6,  0,  0,  0,  0],  # Propyl            +C3H6
      64:  [ 4,  4,  0,  3,  0,  0],  # Succinyl          +C4H4O3
     121:  [ 4,  6,  2,  2,  0,  0],  # GlyGly            +C4H6N2O2 (ubiquitin)
     122:  [ 1,  0,  0,  1,  0,  0],  # Formyl            +CO
     188:  [ 0,  0,  0,  1,  0,  0],  # Hydroxyproline    +O
     198:  [ 8, 13,  1,  5,  0,  0],  # HexNAc            +C8H13NO5
     208:  [ 6, 10,  0,  5,  0,  0],  # Hex               +C6H10O5
     214:  [ 4,  5,  1,  2,  0,  0],  # iTRAQ4plex        +C4H5NO2
     259:  [ 7,  7,  3,  3,  0,  0],  # iTRAQ8plex        +C7H7N3O3
     312:  [ 0, -2,  0, -1,  0,  0],  # Cys->Dha          −H2O
     354:  [ 8, 15,  1,  3,  0,  0],  # TMT2plex          +C8H15NO3
     737:  [ 8, 15,  1,  3,  0,  0],  # TMT6plex          +C8H15NO3
     739:  [ 8, 15,  1,  3,  0,  0],  # TMT10plex         +C8H15NO3
     744:  [ 8, 15,  1,  3,  0,  0],  # TMT11plex         +C8H15NO3
    2016:  [11, 20,  2,  3,  0,  0],  # TMTpro (16-plex)  +C11H20N2O3
}


In [ ]:
### Parsers and Helper functions

#
from functools import lru_cache

from pyteomics.mass import Unimod

# Lazy Unimod loader
_unimod_db = None


def _get_unimod():
    global _unimod_db
    if _unimod_db is None:
        _unimod_db = Unimod()
    return _unimod_db


# ============================================================
# UNIMOD fallback
# ============================================================

@lru_cache(maxsize=512)
def unimod_delta(unimod_id: str):
    """
    Retrieve atomic delta from Unimod via pyteomics.

    Returns [C, H, N, O, S, P]
    """

    # Fast local lookup
    if unimod_id in _UNIMOD_DELTAS:
        return _UNIMOD_DELTAS[unimod_id]

    try:
        db = _get_unimod()

        if "UNIMOD" in str(unimod_id):
            uid = int(unimod_id.replace("UNIMOD:", ""))
        else:
            uid = int(unimod_id)
        mod = db.by_id(uid)
        print(mod)

        composition = mod["composition"]

        return [
            float(composition.get(atom, 0))
            for atom in ATOM_ORDER
        ]

    except Exception as e:
        print(e)
        return [0.0] * N_ATOMS

# ══════════════════════════════════════════════════════════════════════════════
# PARSING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _parse_token(token: str):
    """
    Parse one element of _parsed_sequence.

    Returns (amino_acid: str, unimod_ids: list[int])

    Handles unmodified residues and arbitrarily stacked modifications:
      'K'                        → ('K', [])
      'K[UNIMOD:737]'            → ('K', [737])
      'K[UNIMOD:737][UNIMOD:1]'  → ('K', [737, 1])

    N/C-terminal modifications are NOT parsed here – they live in the
    separate _n_term_mods / _c_term_mods columns and are read by
    _parse_terminal_mod() instead.

    Unknown or empty tokens fall back to ('X', []).
    """
    token = token.strip()
    if not token or token == _PAD_TOKEN:
        return 'X', []

    aa   = token[0].upper()
    uids = []

    # Walk the tail of the token, extracting every [UNIMOD:N] block in order
    rest = token[1:]
    while '[' in rest:
        open_  = rest.index('[')
        close_ = rest.index(']', open_)
        inner  = rest[open_ + 1 : close_]   # e.g. 'UNIMOD:737'
        if ':' in inner:
            uids.append(int(inner.split(':')[1]))
        rest = rest[close_ + 1:]

    return aa, uids


def _parse_terminal_mod(term_string: str):
    """
    Extract Unimod IDs from a terminal modification string.

    Returns list[int]  (empty list = no modification)

    Recognised input formats:
      '[UNIMOD:737]-'            → [737]
      '-[UNIMOD:2]'              → [2]
      '[UNIMOD:1][UNIMOD:737]-'  → [1, 737]   (multiple terminal mods)
      '-'                        → []
      '-[]'                      → []          (empty bracket = no mod)
      ''                         → []

    The function is positionally agnostic: it extracts every [UNIMOD:N]
    present in the string regardless of whether '-' is a prefix or suffix.
    """
    uids = []
    s    = term_string.strip() if term_string else ''
    while '[' in s:
        open_  = s.index('[')
        close_ = s.index(']', open_)
        inner  = s[open_ + 1 : close_]      # e.g. 'UNIMOD:737' or ''
        if ':' in inner:
            uids.append(int(inner.split(':')[1]))
        s = s[close_ + 1:]
    return uids


def _build_atom_vector(aa: str, uids: list) -> list:
    """
    Build [C, H, N, O, S, P] for a residue with zero or more PTM deltas.

    Unknown amino acids produce _PAD_ATOM.
    Unknown Unimod IDs produce a zero delta (modification silently ignored).
    A new list is always returned; the lookup tables are never mutated.
    """
    vec = list(_AA_ATOMS.get(aa, _PAD_ATOM))
    for uid in uids:
        delta = _UNIMOD_DELTAS.get(uid, _PAD_ATOM)
        vec   = [v + d for v, d in zip(vec, delta)]
    return vec


### 1 Custom Feature Extractors for DeepLC

The features are extracted by implementing map functions that receive `inputs` as a dictionary of one record in the dataset, modify it by adding the feature column and referencing existing columns if needed (e.g. sequence) and returning the final updated record. The DLOmix dataset classes use the feature extractors to add the features to the dataset and make them available for model training as tensors.

Existing helpful columns that can be referenced in the dataset are the following, these columns are extracted when parsing the input data and made available in the dataset, they are not meant to be modified by the feature extractors, but rather used to generate features and facilitate parsing the sequence:
- `_parsed_sequence`: the parsed sequence of the record according to the PROFORMA format with amino acid and modifications pairs as one string in a list
  `['Y', 'V', 'S', 'P', 'K[UNIMOD:737]', 'Q', 'F', 'S']`
- `_n_term_mods`: any N-terminal modifications present as a single string
- `_c_term_mods`: any C-terminal modifications present as a single string


In [ ]:
def atoms_per_pos(inputs):
    """
    Per-position atomic composition, zero-padded to _MAX_LEN rows.

    Each row i is [C, H, N, O, S, P] for the residue at position i with all
    PTM deltas (including stacked mods) already summed in.

    N-terminal modification atoms (from _n_term_mods) are added to row 0.
    C-terminal modification atoms (from _c_term_mods) are added to the last
    residue row (index = len(_parsed_sequence) − 1).
    Padding rows (positions ≥ sequence length) remain all zeros.

    Output column : "atoms_per_pos"
    Type          : list[list[int]]
    Shape         : (_MAX_LEN, 6) = (60, 6)

    Example
    ───────
    _parsed_sequence = ['Y', 'V', 'S', 'P', 'K[UNIMOD:737]', 'Q', 'F', 'S']
    _n_term_mods     = '[UNIMOD:1]-'    ← Acetyl  +[2,2,0,1,0,0]
    _c_term_mods     = '-'              ← no mod

    pos 0  Y + N-term Acetyl  : [9,9,1,2,0,0] + [2,2,0,1,0,0] = [11,11,1,3,0,0]
    pos 1  V                  : [ 5,  9, 1, 1, 0, 0]
    pos 2  S                  : [ 3,  5, 1, 2, 0, 0]
    pos 3  P                  : [ 5,  7, 1, 1, 0, 0]
    pos 4  K + TMT6plex (737) : [6,12,2,1,0,0] + [8,15,1,3,0,0] = [14,27,3,4,0,0]
    pos 5  Q                  : [ 5,  8, 2, 2, 0, 0]
    pos 6  F                  : [ 9,  9, 1, 1, 0, 0]
    pos 7  S  (last, no C-mod): [ 3,  5, 1, 2, 0, 0]
    pos 8–59                  : [ 0,  0, 0, 0, 0, 0]  ← padding
    """
    seq = inputs['_parsed_sequence']
    n   = min(len(seq), _MAX_LEN)
    out = [list(_PAD_ATOM) for _ in range(_MAX_LEN)]

    # ── Residue-level atoms ───────────────────────────────────────────────
    for i, token in enumerate(seq[:n]):
        aa, uids = _parse_token(token)
        out[i]   = _build_atom_vector(aa, uids)

    # ── N-terminal modification → fold into position 0 ───────────────────
    if n > 0:
        for uid in _parse_terminal_mod(inputs.get('_n_term_mods', '')):
            delta  = unimod_delta(uid)
            #delta  = _UNIMOD_DELTAS.get(uid, _PAD_ATOM)
            out[0] = [v + d for v, d in zip(out[0], delta)]

    # ── C-terminal modification → fold into last residue position ─────────
    if n > 0:
        last = n - 1
        for uid in _parse_terminal_mod(inputs.get('_c_term_mods', '')):
            delta  = unimod_delta(uid)
            #delta     = _UNIMOD_DELTAS.get(uid, _PAD_ATOM)
            out[last] = [v + d for v, d in zip(out[last], delta)]

    inputs['atoms_per_pos'] = out
    return inputs

In [ ]:
def diaa_atoms(inputs):
    """
    Di-amino acid atomic composition: non-overlapping adjacent pairs of
    residues summed into one atom vector per pair.

    Pairs: (pos 0, pos 1), (pos 2, pos 3), ..., (_MAX_LEN // 2 pairs total).
    Each pair vector = atom_vector[pos_a] + atom_vector[pos_b].
    Odd-length sequences leave the last pair's second slot as zero.
    Padding pair rows are all zeros.

    N/C-terminal modification atoms are folded into position 0 and the last
    residue position respectively before pairing, consistent with atoms_per_pos.

    Output column : "diaa_atoms"
    Type          : list[list[int]]
    Shape         : (_MAX_LEN // 2, 6) = (30, 6)

    Example
    ───────
    _parsed_sequence = ['Y', 'V', 'S', 'P', 'K[UNIMOD:737]', 'Q', 'F', 'S']
    _n_term_mods     = '-'
    _c_term_mods     = '-'

    pair 0  (Y, V)      : [ 9,9,1,2,0,0] + [5,9,1,1,0,0]       = [14,18,2,3,0,0]
    pair 1  (S, P)      : [ 3,5,1,2,0,0] + [5,7,1,1,0,0]       = [ 8,12,2,3,0,0]
    pair 2  (K+TMT, Q)  : [14,27,3,4,0,0] + [5,8,2,2,0,0]      = [19,35,5,6,0,0]
    pair 3  (F, S)      : [ 9,9,1,1,0,0] + [3,5,1,2,0,0]       = [12,14,2,3,0,0]
    pairs 4–29          : [0, 0, 0, 0, 0, 0]
    """
    seq     = inputs['_parsed_sequence']
    n       = min(len(seq), _MAX_LEN)
    n_pairs = _MAX_LEN // 2   # = 30

    # ── Build per-position vectors ────────────────────────────────────────
    per_pos = [list(_PAD_ATOM) for _ in range(_MAX_LEN)]

    for i, token in enumerate(seq[:n]):
        aa, uids   = _parse_token(token)
        per_pos[i] = _build_atom_vector(aa, uids)

    # N-terminal mod → position 0
    if n > 0:
        for uid in _parse_terminal_mod(inputs.get('_n_term_mods', '')):
            delta  = unimod_delta(uid)
            #delta      = _UNIMOD_DELTAS.get(uid, _PAD_ATOM)
            per_pos[0] = [v + d for v, d in zip(per_pos[0], delta)]

    # C-terminal mod → last residue position
    if n > 0:
        last = n - 1
        for uid in _parse_terminal_mod(inputs.get('_c_term_mods', '')):
            delta  = unimod_delta(uid)
            #delta          = _UNIMOD_DELTAS.get(uid, _PAD_ATOM)
            per_pos[last]  = [v + d for v, d in zip(per_pos[last], delta)]

    # ── Sum non-overlapping pairs ─────────────────────────────────────────
    out = [list(_PAD_ATOM) for _ in range(n_pairs)]

    for pair_idx in range(n_pairs):
        pos_a = pair_idx * 2
        pos_b = pair_idx * 2 + 1
        vec   = list(_PAD_ATOM)
        if pos_a < n:
            vec = [v + p for v, p in zip(vec, per_pos[pos_a])]
        if pos_b < n:
            vec = [v + p for v, p in zip(vec, per_pos[pos_b])]
        out[pair_idx] = vec

    inputs['diaa_atoms'] = out
    return inputs

In [ ]:
def global_feats(inputs):
    """
    Peptide-level global (general) feature vector for the DeepLC Cerberus model.

    Produces a 55-element flat list that concatenates:
      1. Whole-peptide atom totals  (6 values)
      2. Sequence length            (1 value)
      3. Per-residue atom vectors for the first 4 residues and last 4 residues
         of the peptide, each with per-residue PTMs included but WITHOUT any
         terminal modification atoms  (8 × 6 = 48 values)

    Total: 6 + 1 + 48 = 55 values.

    Parameters
    ----------
    inputs : dict
        Must contain:
          '_parsed_sequence' : list[str]
              ProForma-style tokens, e.g. ['Y', 'K[UNIMOD:737]', 'S', ...]
          '_n_term_mods'     : str  (optional)
              N-terminal modification string, e.g. '[UNIMOD:1]-'
          '_c_term_mods'     : str  (optional)
              C-terminal modification string, e.g. '-[UNIMOD:2]'

    Side effect
    -----------
    Stores the 55-vector under inputs['global_feats'] and returns inputs.

    Example
    ───────
    _parsed_sequence = ['Y', 'V', 'S', 'P', 'K[UNIMOD:737]', 'Q', 'F', 'S']
    _n_term_mods     = '[UNIMOD:1]-'   ← Acetyl  +[2,2,0,1,0,0]
    _c_term_mods     = '-'

    Whole-peptide totals (including N-term Acetyl):
      Y:             [ 9,  9, 1, 2, 0, 0]
      V:             [ 5,  9, 1, 1, 0, 0]
      S:             [ 3,  5, 1, 2, 0, 0]
      P:             [ 5,  7, 1, 1, 0, 0]
      K+TMT6plex:    [14, 27, 3, 4, 0, 0]
      Q:             [ 5,  8, 2, 2, 0, 0]
      F:             [ 9,  9, 1, 1, 0, 0]
      S:             [ 3,  5, 1, 2, 0, 0]
      N-term Acetyl: [ 2,  2, 0, 1, 0, 0]
      ─────────────────────────────────────
      Σ:             [55, 81,11,16, 0, 0]

    seq_len = 8

    Positional block (residue-only atoms, no terminal mods):
      pos  0  Y:             [ 9,  9, 1, 2, 0, 0]
      pos  1  V:             [ 5,  9, 1, 1, 0, 0]
      pos  2  S:             [ 3,  5, 1, 2, 0, 0]
      pos  3  P:             [ 5,  7, 1, 1, 0, 0]
      pos -4  K+TMT6plex:    [14, 27, 3, 4, 0, 0]
      pos -3  Q:             [ 5,  8, 2, 2, 0, 0]
      pos -2  F:             [ 9,  9, 1, 1, 0, 0]
      pos -1  S:             [ 3,  5, 1, 2, 0, 0]

    Full output (55 values):
      [55, 81, 11, 16, 0, 0,   ← whole-peptide atom totals
       8,                       ← seq_len
       9,  9, 1, 2, 0, 0,      ← pos  0  (Y)
       5,  9, 1, 1, 0, 0,      ← pos  1  (V)
       3,  5, 1, 2, 0, 0,      ← pos  2  (S)
       5,  7, 1, 1, 0, 0,      ← pos  3  (P)
       14,27, 3, 4, 0, 0,      ← pos -4  (K+TMT6plex)
       5,  8, 2, 2, 0, 0,      ← pos -3  (Q)
       9,  9, 1, 1, 0, 0,      ← pos -2  (F)
       3,  5, 1, 2, 0, 0]      ← pos -1  (S)
    """
    seq = inputs['_parsed_sequence']
    n   = min(len(seq), _MAX_LEN)

    # ── 1. Build per-position atom vectors (NO terminal mods) ─────────────
    # These are used for both the positional block and the global totals.
    per_pos = [list(_PAD_ATOM) for _ in range(n)]
    for i, token in enumerate(seq[:n]):
        aa, uids  = _parse_token(token)
        per_pos[i] = _build_atom_vector(aa, uids)

    # ── 2. Whole-peptide atom totals  (includes terminal mods) ────────────
    total = [sum(col) for col in zip(*per_pos)] if n > 0 else list(_PAD_ATOM)

    # Add N-terminal modification atoms
    for uid in _parse_terminal_mod(inputs.get('_n_term_mods', '')):
        delta = unimod_delta(uid)
        total = [t + d for t, d in zip(total, delta)]

    # Add C-terminal modification atoms
    for uid in _parse_terminal_mod(inputs.get('_c_term_mods', '')):
        delta = unimod_delta(uid)
        total = [t + d for t, d in zip(total, delta)]

    # ── 3. Sequence length ─────────────────────────────────────────────────
    seq_len = n  # raw residue count, capped at _MAX_LEN

    # ── 4. Positional block: first 4 and last 4 residues ─────────────────
    # Positive indices: absolute positions 0–3 from the N-terminal end.
    # Negative indices: -4 to -1 mapped to (n-4)..(n-1) from the C-terminal end.
    # For short peptides (n < 4) the missing slots remain _PAD_ATOM.
    pos_atoms = []

    for p in _POS_INDICES:
        if p < n:
            pos_atoms.extend(per_pos[p])
        else:
            pos_atoms.extend(_PAD_ATOM)

    for p in _NEG_INDICES:
        abs_idx = n + p          # e.g. n=8, p=-4  →  abs_idx=4
        if abs_idx >= 0:
            pos_atoms.extend(per_pos[abs_idx])
        else:
            pos_atoms.extend(_PAD_ATOM)

    # ── 5. Assemble final vector ───────────────────────────────────────────
    # [6 atom totals | 1 seq_len | 48 positional atoms] = 55 values
    global_vec = total + [seq_len] + pos_atoms

    inputs['global_feats'] = global_vec
    return inputs


In [ ]:
feature_functions = [
    atoms_per_pos, diaa_atoms, global_feats
]

### 1 PROSPECT DATA Subset from HuggingFace

Load the Prospect PTM dataset from Hugging Face Hub directly and extract the DeepLC input features.

In [ ]:
### full prospect retention time dataset
# for shorter runs, please subset the data if needed

# Dataset preparation, feature extraction - this would take time
from dlomix.data import RetentionTimeDataset

dataset = RetentionTimeDataset(
    data_format="hub",
    data_source="Wilhelmlab/prospect-ptms-irt",
    sequence_column="modified_sequence",
    label_column="indexed_retention_time",
    # add pre-comupted features here, not functions, we do not have any now
    #model_features=[],
    # add functions here, not pre-comupted column names, we have all our features here as functions
    features_to_extract=feature_functions,
    max_seq_len=60,
    alphabet=None,
    encoding_scheme="unmod",
    batch_size=128,
    with_termini=False,
    enable_tf_dataset_cache=True,
    dataset_type="tf",
    padding_value=_PAD_TOKEN,
)


In [ ]:
dataset

In [ ]:
# validate shapes of features

import numpy as np
("atom counts", np.array(atoms_per_pos(dataset["train"][0])["atoms_per_pos"]).shape,
 "---",
"diaatom counts", np.array(diaa_atoms(dataset["train"][0])["diaa_atoms"]).shape,
 "---",
"global features", np.array(global_feats(dataset["train"][0])["global_feats"]).shape)

In [ ]:
dataset["train"]["_parsed_sequence"][:5]

### 3 Model

Create the DeepLC model, use the alphabet from the dataset and pass the string keys for the inputs.

In [ ]:
from dlomix.models import DeepLCRetentionTimePredictor
model = DeepLCRetentionTimePredictor(
    seq_length=_MAX_LEN,
    use_global_features=True,
    alphabet=dataset.extended_alphabet,
    sequence_input_key="modified_sequence",
    counts_input_key="atoms_per_pos",
    di_counts_input_key="diaa_atoms",
    global_features_input_key="global_feats",
)

In [ ]:
import tensorflow as tf
from dlomix.eval import timedelta

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.MeanAbsoluteError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
        tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        timedelta,
    ],
)

In [ ]:
# Model training - this would take time

history = model.fit(
    dataset.tensor_train_data,
    validation_data=dataset.tensor_val_data,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae",
            patience=10,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
        ),
    ],
)

Next: evaluate the predictions or save the trained model.

In [ ]:
predictions = model.predict(dataset.tensor_test_data)
predictions.shape